# Forecasting 1 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Urban convenience stores

**Problem:** Forecast next-week sales per store to improve replenishment planning.

**Proposed success criterion:** Beat the seasonal-naive forecast MAE by at least 10% on the last 12 weeks; examine store-level errors.

**Dataset:** `forecasting_1.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


## Phase 1 — Business Understanding and Phase 2 — Data Understanding (review)
Explain the decision, stakeholder, prediction moment, observation unit, key and target. **Review or revise the supplied success criterion if it does not adequately represent the business decision.** State two evidenced findings and two limitations.

The decision is how much stock should be replenished for each store for the following week. The main stakeholder is the operations/replenishment manager, who uses the forecast to plan inventory. The prediction moment is before the start of the next week, when replenishment decisions must be made. One observation represents one store in one week. The business key is store_id + week_start, and the target is sales_eur.

The proposed success criterion, improving the seasonal, naive MAE by at least 10% on the last 12 weeks — is measurable and useful, but it does not fully represent the business decision. It should also evaluate errors per store and distinguish between underforecasting, which can cause stockouts, and overforecasting, which can create excess inventory.

Two evidenced findings:
1. Average weekly sales are about €10,534, but the maximum is €82,000, indicating a possible outlier.
2. 20/05/2024 occurs 9 times while there are only 8 stores, indicating a possible duplicate store-week observation.
3. There is a overlapping between the variables of store_floor_m2 and store_size_category.

Two limitations:
1. footfall_actual and rain_mm_actual are actual values and may not be known when the next-week forecast is made.
2. returns_eur_after_week is only known after the week and could therefore cause data leakage.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('forecasting_1.csv')
if not DATA.exists(): DATA = Path('datasets/forecasting_1.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(841, 11)


,store_id,week_start,promo_label,holiday_week,footfall_actual,rain_mm_actual,store_floor_m2,store_size_category,average_price_eur,returns_eur_after_week,sales_eur
0,S01,01/01/2024,No,0,366,0.0,504,Large,12.13,280.49,9657.59
1,S01,08/01/2024,No,0,352,13.4,504,Large,12.05,465.21,9785.84
2,S01,15/01/2024,Yes,0,314,9.6,504,Large,13.96,535.72,11873.38
3,S01,22/01/2024,No,0,324,0.0,504,Large,13.97,482.79,10125.01
4,S01,29/01/2024,No,0,323,7.8,504,Large,12.05,451.27,10080.88


store_id                  0
week_start                0
promo_label               0
holiday_week              0
footfall_actual           0
rain_mm_actual            0
store_floor_m2            0
store_size_category       0
average_price_eur         0
returns_eur_after_week    0
sales_eur                 0
dtype: int64

Exact duplicates: 1


## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [7]:
# Add your preparation code below. Inspect before/after evidence.
# Make a copy first:
work = df.copy()

# Evidence before
print("Duplicate store-week keys:",
      work.duplicated(subset=['store_id', 'week_start']).sum())

duplicates = work[
    work.duplicated(subset=['store_id', 'week_start'], keep=False)
]

display(duplicates.sort_values(['store_id', 'week_start']))

before = len(work)

work = work.drop_duplicates()

print("Rows before:", before)
print("Rows after:", len(work))

Duplicate store-week keys: 1


,store_id,week_start,promo_label,holiday_week,footfall_actual,rain_mm_actual,store_floor_m2,store_size_category,average_price_eur,returns_eur_after_week,sales_eur
125,S02,20/05/2024,No,0,491,1.3,654,Large,13.8,618.54,12998.75
126,S02,20/05/2024,No,0,491,1.3,654,Large,13.8,618.54,12998.75


Rows before: 841
Rows after: 840


In [9]:
print("Before:", work.columns.tolist())

work = work.drop(columns=['returns_eur_after_week'])

print("After:", work.columns.tolist())

Before: ['store_id', 'week_start', 'promo_label', 'holiday_week', 'footfall_actual', 'rain_mm_actual', 'store_floor_m2', 'store_size_category', 'average_price_eur', 'returns_eur_after_week', 'sales_eur', 'month', 'week_number', 'year']
After: ['store_id', 'week_start', 'promo_label', 'holiday_week', 'footfall_actual', 'rain_mm_actual', 'store_floor_m2', 'store_size_category', 'average_price_eur', 'sales_eur', 'month', 'week_number', 'year']


### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
|A duplicate store-week key was found|Checked store_id + week_start for duplicates and removed an exact duplicate. |The number of duplicate records decreased to zero.|Each row should represent one store in one week.|Use store_id + week_start as the business key. |
|week_start was stored as text|The datatype changed from object/string to datetime.  |Converted it to datetime and derived month, week number and year. |This makes time patterns and seasonality usable.|Retain the derived time features for modelling. |
|returns_eur_after_week is only available after the forecast week.|Removed the variable from the predictor set. |The column is no longer present in the prepared dataset. |Using it would give the model information that is unavailable at the prediction moment. |Exclude it from all forecasting models. |

## Phase 4 — Modelling
Define the prediction moment and allowed predictors. Use a defensible train/test split (chronological for forecasting). Fit learned preprocessing on TRAINING data only. Compare a simple baseline with a candidate model. Explain your feature engineering and excluded fields.

1. 

Target: sales_eur, because the goal is to predict weekly sales per store.

Prediction moment: The prediction is made before the start of the next week, so the replenishment manager can plan inventory.

Eligible predictors: store_id, promo_label, holiday_week, store_floor_m2 and historical sales values. average_price_eur can only be used if the price is known before the week starts.

Excluded predictors: returns_eur_after_week is excluded because it becomes available after the sales week. footfall_actual and rain_mm_actual are excluded because their actual values are not known at the prediction moment. store_size_category is removed because it overlaps with store_floor_m2.

2.

A chronological train/test split was used because this is a forecasting problem. The earlier weeks were used for training and the final 12 weeks were used as the test set.

This represents the real business situation because the model learns from historical data and predicts future weeks. A random split was avoided because it could allow future observations to influence predictions of earlier weeks.

The duplicate store-week record was removed before modelling, so the same observation cannot occur in both the training and test data.

3.
| Baseline and why | Candidate and why | Test method | Result / interpretation |
|---|---|---|---|
| Seasonal-naive forecast using sales from the same store 52 weeks earlier. This is a suitable baseline because weekly sales may show yearly seasonality. | Linear Regression using store, promotion, holiday, store size and historical sales. This model is simple and can combine multiple predictors. | A chronological split was used. The model was trained on earlier weeks and tested on the final 12 weeks. MAE was used to compare both models. | The Linear Regression model performed 5.1% worse than the seasonal-naive baseline. Therefore, the success criterion of at least 10% improvement was not achieved. |

In [17]:
# Build and test your baseline and candidate model here.
# Suggested tools: sklearn Pipeline, ColumnTransformer, SimpleImputer,
# OneHotEncoder, LogisticRegression, RandomForestRegressor or Ridge.
# For forecasting: use time-ordered validation; no future sales or actual future footfall.

work = work.sort_values(['store_id', 'week_start'])

# Sales from the same week last year
work['sales_lag_52'] = work.groupby('store_id')['sales_eur'].shift(52)

weeks = sorted(work['week_start'].unique())

train = work[work['week_start'].isin(weeks[:-12])]
test = work[work['week_start'].isin(weeks[-12:])]

features = [
    'store_id',
    'promo_label',
    'holiday_week',
    'store_floor_m2',
    'sales_lag_52'
]

target = 'sales_eur'

train = train.dropna(subset=['sales_lag_52'])
test = test.dropna(subset=['sales_lag_52'])

X_train = pd.get_dummies(train[features], columns=['store_id', 'promo_label'])
X_test = pd.get_dummies(test[features], columns=['store_id', 'promo_label'])

# Make sure train and test have the same columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train[target]
y_test = test[target]

from sklearn.metrics import mean_absolute_error

baseline_mae = mean_absolute_error(
    y_test,
    test['sales_lag_52']
)

print("Baseline MAE:", baseline_mae)

Baseline MAE: 974.7604166666666


In [22]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

model_mae = mean_absolute_error(y_test, predictions)

print("Model MAE:", model_mae)

Model MAE: 1024.753899068057


In [23]:
improvement = (baseline_mae - model_mae) / baseline_mae * 100

print("Improvement:", round(improvement, 1), "%")

Improvement: -5.1 %


## Phase 5 — Evaluation
Report model vs baseline on held-out observations. Interpret relevant classification errors or forecasting MAE by business segment. Compare with your Phase 1 success criterion and discuss whether that criterion is realistic. If a required cost/impact is unavailable, state the assumption or request more data.

## Model Evaluation

The model was evaluated on the **final 12 weeks**, containing 96 held-out observations across 8 stores. These observations were not used to train the model.

| Evaluation | Seasonal-naive baseline | Linear Regression |
|---|---:|---:|
| Overall MAE | €974.76 | €1,024.75 |
| Difference | - | 5.1% worse |

The Linear Regression model has a higher MAE than the seasonal-naive baseline. Therefore, the candidate model does not outperform the baseline.

### MAE by store

| Store | Baseline MAE | Model MAE | Change |
|---|---:|---:|---:|
| S01 | €1,021.95 | €1,037.71 | -1.5% |
| S02 | €1,012.78 | €920.67 | +9.1% |
| S03 | €691.90 | €1,977.79 | -185.9% |
| S04 | €836.15 | €728.07 | +12.9% |
| S05 | €1,188.99 | €966.86 | +18.7% |
| S06 | €748.27 | €610.43 | +18.4% |
| S07 | €1,425.31 | €1,122.07 | +21.3% |
| S08 | €872.72 | €834.43 | +4.4% |

The model performs better than the baseline for most stores, especially S04, S05, S06 and S07. However, performance for S03 is much worse. This large error strongly increases the overall MAE and should be investigated before the model is used.

### Comparison with Phase 1 success criterion

The Phase 1 success criterion was to reduce the seasonal-naive MAE by at least 10% on the final 12 weeks. This criterion was **not achieved**. Instead, the overall MAE increased by 5.1%.

The 10% target may be ambitious for the current Linear Regression model and available predictors. However, one model is not enough to conclude that the criterion itself is unrealistic. Other models or additional information available before the forecast week could still improve performance.

### Business impact limitation

The financial cost of forecasting too high or too low is not available in the dataset. Therefore, MAE is currently treated as if overforecasting and underforecasting have equal cost. In practice, underforecasting may cause stockouts and lost sales, while overforecasting may cause excess inventory. More information about these costs is needed to evaluate the true business impact of forecast errors.

## Phase 6 — Deployment proposal (design only)
Specify who uses the prediction, when it runs, what action it supports, a human override, monitoring metric, feedback/refresh trigger and a condition for pausing use. A diagram or 6–8 sentences is sufficient. No production system required.

The weekly sales forecast would be used by the **replenishment manager** to support inventory planning for each store.  
The forecast would run **once per week before the start of the next sales week**, using the most recent available data.  
The predicted `sales_eur` would help the manager decide how much stock should be replenished for each store.  
The forecast should be used as **decision support**, meaning the manager can override the prediction when additional information is available, such as local events or unusual promotions.  
Forecast performance should be monitored using **MAE overall and MAE per store**, because performance can differ between locations.  
The model should be reviewed or retrained when new weekly sales data becomes available and when forecast errors start increasing.  
Use of the model should be paused if its MAE becomes consistently worse than the seasonal-naive baseline or if important input data is missing or unreliable.  
Because the Linear Regression model did not achieve the 10% improvement criterion, it should first be tested further before operational deployment.

## Final group decision + individual accountability
**Group:** Pilot / Improve / Revisit earlier CRISP-DM phase / Do not deploy yet. Cite evidence.

**Individual:** Every member adds a named Markdown paragraph describing one preparation choice, one model finding and one business implication.